# L5 demo: a 4-stage SECOM pipeline, three ways

This notebook builds the same four-stage batch pipeline (**ingest → clean → impute →
aggregate**) on the UCI SECOM dataset three times: once in pandas, once in Polars
lazy, and once in Dask across partitions. Each version is a single reproducible
function so it can be timed and rerun. The pandas and Polars runs should agree on
every number; the Dask run exists to make the out-of-core execution model concrete,
not because this dataset needs it.

SECOM is 1,567 semiconductor manufacturing runs with 590 sensor/process
measurements each, heavy missingness, several constant columns, and a pass/fail
label. It is small enough to fit in memory on a laptop, which is itself part of
the lesson: watch for the point where Dask's overhead outweighs its benefit.

## Fetch and cache the data

The raw files are cached under `.cache/` on first run, exactly as in the L1 and L3
demos, so re-running this notebook does not re-download anything.

In [ ]:
import io
import urllib.request
import warnings
import zipfile
from pathlib import Path

import pandas as pd
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
# A 590-wide dataframe built by concatenation trips pandas' block-fragmentation
# heuristic on every later column write. Harmless here; silenced so the signal
# is not lost in the noise for the operations that actually matter.

CACHE = Path('.cache')
CACHE.mkdir(exist_ok=True)

DATA_FILE = CACHE / 'secom.data'
LABELS_FILE = CACHE / 'secom_labels.data'
URL = 'https://archive.ics.uci.edu/static/public/179/secom.zip'

if not DATA_FILE.exists():
    print('downloading', URL)
    with urllib.request.urlopen(URL) as r:
        payload = r.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as z:
        DATA_FILE.write_bytes(z.read('secom.data'))
        LABELS_FILE.write_bytes(z.read('secom_labels.data'))


## Stage 1: ingest

`secom.data` is 590 single-space-separated columns with no header and `NaN` for
missing values. `secom_labels.data` carries the pass/fail label and a collection
timestamp per run, in the same row order. We tag each run with a `run_id` so
every later stage can be traced back to a specific wafer run.

The labels file hides a real footgun. A line of it looks like this:

```
-1 "19/07/2008 11:55:00"
```

Two fields, not three: the timestamp is a *single quoted field* that happens to
contain a space. Read it with `sep=r'\s+'` and `names=['label', 'date', 'time']`
and pandas honours the quote character, hands you the whole timestamp as `date`,
finds no third field, and fills `time` with `NaN`. Nothing raises. You then get
a `ts` column that is entirely `NaT`, and stage 4's "daily mean" silently
collapses 86 days of production into one meaningless bucket. Read the raw bytes
of a file before you write the parser for it.

In [ ]:
import numpy as np
import pandas as pd

def ingest_pandas(data_file=DATA_FILE, labels_file=LABELS_FILE):
    X = pd.read_csv(data_file, sep=r'\s+', header=None, na_values='NaN')
    X.columns = [f'sensor_{i}' for i in range(X.shape[1])]

    # Two fields per line: the label, and the timestamp as one quoted field.
    lab = pd.read_csv(labels_file, sep=r'\s+', header=None, names=['label', 'ts'])
    lab['ts'] = pd.to_datetime(lab['ts'], format='%d/%m/%Y %H:%M:%S')

    raw = pd.concat([lab[['ts', 'label']], X], axis=1)
    raw = pd.concat([pd.Series(np.arange(len(raw)), name='run_id'), raw], axis=1)
    return raw.copy()  # one contiguous block; avoids fragmentation on later column writes

raw = ingest_pandas()
sensor_cols = [c for c in raw.columns if c.startswith('sensor_')]

# Assert the parse worked, rather than trusting that it did.
assert raw['ts'].notna().all(), 'timestamp parse failed; check the quoting'
print(raw.shape, 'runs x columns;', len(sensor_cols), 'sensors')
print('runs span', raw['ts'].min(), 'to', raw['ts'].max(),
      f"({raw['ts'].dt.date.nunique()} distinct days)")
print('labels:', dict(raw['label'].value_counts()), '(-1 = pass, 1 = fail)')
print(f"overall missing fraction: {raw[sensor_cols].isna().mean().mean():.2%}")
raw.iloc[:3, :6]


## Stage 2: clean

Drop columns that are constant (zero information) or missing beyond a
threshold, computed on the **training split only** -- fitting this on the full
dataset, test rows included, is exactly the leakage bug A3 asks you to avoid.

In [ ]:
def clean_pandas(df, sensor_cols, missing_frac=0.4):
    """Return the trimmed frame and the columns it dropped."""
    nunique = df[sensor_cols].nunique(dropna=True)
    constant = nunique[nunique <= 1].index
    miss = df[sensor_cols].isna().mean()
    high_missing = miss[miss > missing_frac].index
    drop_cols = sorted(set(constant) | set(high_missing))
    return df.drop(columns=drop_cols), drop_cols

n_train = int(len(raw) * 0.8)
train, test = raw.iloc[:n_train].copy(), raw.iloc[n_train:].copy()

train_clean, dropped = clean_pandas(train, sensor_cols)
keep_cols = [c for c in train_clean.columns if c.startswith('sensor_')]
test_clean = test.drop(columns=dropped)

# Report the two reasons separately. "150 columns dropped" hides the fact that
# most of them carry no information at all rather than merely being sparse.
nu = train[sensor_cols].nunique(dropna=True)
ms = train[sensor_cols].isna().mean()
n_const = int((nu <= 1).sum())
n_sparse = int((ms > 0.4).sum())
print(f'{len(dropped)} of {len(sensor_cols)} sensor columns dropped, {len(keep_cols)} remain')
print(f'  constant (<=1 distinct value): {n_const}')
print(f'  more than 40% missing:         {n_sparse}')
print(f'  (the two sets overlap by {n_const + n_sparse - len(dropped)})')


## Stage 3: impute

Fill missing sensor readings with the **training mean**, then apply that same,
frozen statistic to the test split. Computing the mean on the test rows too, or
on the whole dataset before splitting, is the single most common way this kind
of pipeline leaks the future into the past.

In [ ]:
def impute_pandas(reference, target, cols):
    means = reference[cols].mean()
    out = target.copy()
    out[cols] = out[cols].fillna(means)
    return out

train_imputed = impute_pandas(train_clean, train_clean, keep_cols)
test_imputed = impute_pandas(train_clean, test_clean, keep_cols)
assert train_imputed[keep_cols].isna().sum().sum() == 0
assert test_imputed[keep_cols].isna().sum().sum() == 0


## Stage 4: aggregate and persist

A daily mean per sensor, cached to Parquet. Writing the same input twice
produces the same file, which is the whole point of keeping every stage a
pure function of its input: the pipeline is idempotent by construction, not
by careful bookkeeping.

In [ ]:
import hashlib

def aggregate_pandas(df, cols):
    return df.assign(day=df['ts'].dt.date).groupby('day')[cols].mean()

daily = aggregate_pandas(train_imputed, keep_cols)
print(f'daily aggregate: {daily.shape[0]} days x {daily.shape[1]} sensors')

out_path = CACHE / 'train_imputed.parquet'
train_imputed.to_parquet(out_path, index=False)
print('wrote', out_path, train_imputed.shape)

# Don't assert idempotency, check it. This is the reconciliation step that the
# Knight Capital case in the notes is an argument for: verify that the batch job
# actually produced what you expected, rather than trusting that it completed.
def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:16]

first = digest(out_path)
train_imputed.to_parquet(out_path, index=False)
second = digest(out_path)
print(f'sha256 (truncated) run 1: {first}\n                    run 2: {second}')
print('byte-for-byte identical:', first == second)
daily.iloc[:5, :4]


## The same four stages in Polars, lazy

One lazy query, described end to end and only executed at `.collect()`. Polars
computes the per-column statistics clean and impute both need -- distinct
count, missing count, and mean -- in a **single pass** over the training
partition, instead of the three separate passes the pandas version above
makes.

In [ ]:
import polars as pl

def secom_polars_pipeline(data_file=DATA_FILE, labels_file=LABELS_FILE,
                           missing_frac=0.4, train_frac=0.8):
    # Declare the schema instead of letting Polars infer it. Every sensor column
    # is a float; saying so is both faster and safer than any inference setting.
    # See the pitfall note below for what happens if you let it guess.
    n_cols = len(data_file.read_text().split('\n', 1)[0].split(' '))
    schema = {f'column_{i + 1}': pl.Float64 for i in range(n_cols)}
    X = pl.read_csv(data_file, separator=' ', has_header=False,
                    null_values=['NaN'], schema_overrides=schema)
    X.columns = [f'sensor_{i}' for i in range(X.width)]

    # Two columns, not three: the timestamp is one quoted field.
    lab = pl.read_csv(labels_file, separator=' ', has_header=False,
                       new_columns=['label', 'ts'])
    lab = lab.with_columns(
        pl.col('ts').str.strptime(pl.Datetime, '%d/%m/%Y %H:%M:%S')
    )

    lf = X.with_row_index('run_id').with_columns([lab['ts'], lab['label']]).lazy()
    sensor_cols = [c for c in lf.collect_schema().names() if c.startswith('sensor_')]

    n_total = lf.select(pl.len()).collect().item()
    n_train = int(n_total * train_frac)
    train_lf = lf.slice(0, n_train)
    test_lf = lf.slice(n_train, n_total - n_train)

    # clean() and impute() share this one pass over the training partition.
    stats = train_lf.select(
        [pl.col(c).drop_nulls().n_unique().alias(f'{c}__nunique') for c in sensor_cols]
        + [pl.col(c).null_count().alias(f'{c}__nmiss') for c in sensor_cols]
        + [pl.col(c).mean().alias(f'{c}__mean') for c in sensor_cols]
        + [pl.len().alias('__n')]
    ).collect()
    n_train_rows = stats['__n'][0]

    keep = [
        c for c in sensor_cols
        if stats[f'{c}__nunique'][0] > 1
        and stats[f'{c}__nmiss'][0] / n_train_rows <= missing_frac
    ]
    means = {c: stats[f'{c}__mean'][0] for c in keep}

    def clean_and_impute(lfr):
        return (
            lfr.select(['run_id', 'ts', 'label'] + keep)
            .with_columns([pl.col(c).fill_null(means[c]) for c in keep])
        )

    train_out = clean_and_impute(train_lf)
    test_out = clean_and_impute(test_lf)
    daily_out = (
        train_out.with_columns(pl.col('ts').dt.date().alias('day'))
        .group_by('day').agg([pl.col(c).mean() for c in keep]).sort('day')
    )
    return train_out.collect(), test_out.collect(), daily_out.collect(), keep

train_pl, test_pl, daily_pl, keep_pl = secom_polars_pipeline()
print(f'{len(keep_pl)} sensors kept, {590 - len(keep_pl)} dropped -- matches the pandas run')
assert set(keep_pl) == set(keep_cols)

# Same columns is necessary but not sufficient: check the imputed values agree too.
max_diff = np.nanmax(np.abs(train_imputed[keep_cols].to_numpy()
                            - train_pl.select(keep_cols).to_numpy()))
print(f'largest disagreement between the two imputed matrices: {max_diff:.3g}')
assert max_diff < 1e-9
print(f'daily aggregate: {daily_pl.height} days, same as pandas: '
      f'{daily_pl.height == daily.shape[0]}')


:::{admonition} Two pitfalls in that one function
:class: warning

**`n_unique()` counts nulls.** `clean_pandas` above uses `nunique(dropna=True)`
to find constant columns, and that `dropna` is load-bearing. Polars'
`Series.n_unique()` counts a null as a distinct value, so a column that is one
constant value plus a scatter of missing readings reports **two** unique values,
not one, and survives a naive port of the pandas rule. The fix is
`pl.col(c).drop_nulls().n_unique()`, used above.

**Schema inference is a guess, and here it guesses wrong.** Leave the dtypes to
Polars and this cell does not run at all:

```
ComputeError: could not parse `4.1955` as dtype `i64` at column 'column_75'
```

Polars samples the first 100 rows by default. The 75th column (`sensor_74` once
we rename) holds whole numbers, almost all of them exactly `0`, for its first
1,457 runs, so `i64` looks right; then `4.1955` arrives on row **1,458** of
1,567, roughly 93% of the way through the file, and it raises. No sane sample
size would have caught that.

There are two ways out, and they are not equally good. Reading this file on the
machine these notes were written on:

| approach | read time | outcome |
|---|---|---|
| default inference (100 rows) | – | **raises** on row 1,458 |
| `infer_schema_length=None` | ~110 ms | works, scans the whole file to decide |
| `schema_overrides={...: pl.Float64}` | **~38 ms** | works, decides nothing |
| pandas `read_csv`, for reference | ~75 ms | works, silently upcasts |

Scanning the entire file to infer what you already know is the expensive way to
be safe: it costs half again as much as pandas' whole read. Declaring the schema
is both the fastest option, by a factor of three, and the only one that cannot be
surprised by row 1,458, which is why the pipeline above does that. It also
documents in code what you believe about the data, and a belief written down is a
belief that can be checked. That idea, schema as an executable artifact rather
than a runtime guess, is exactly where L6 picks up with pandera.

Note the contrast with pandas, which infers types too but silently upcasts
instead of raising. Polars' strictness converts a quiet coercion into a loud
error. Loud is better, but only if you know what the error means.
:::

## Benchmark: pandas vs. Polars, the whole pipeline

Both functions read the raw files and run all four stages, so this times the
thing you actually care about end to end, not one operation in isolation.

Set your expectations before you run it. This is a 1,567-row file, and a large
share of both runtimes is parsing 590 columns of CSV. Expect Polars to come in
around **twice** as fast, not the ten or twenty that benchmark headlines quote
for grouped aggregations on tens of millions of rows. Where does the factor of
two come from? Two places, measured separately above and below: the read is
about 1.5× faster with a declared schema, and the three per-column statistics
that pandas computes in three passes over the frame are computed by Polars in
one, which on this data is closer to 4×.

That is the honest number at this scale, and it is worth sitting with. The
speedup you read about is a speedup on somebody else's data shape. Measure
yours.

In [ ]:
import time

def secom_pandas_pipeline(data_file=DATA_FILE, labels_file=LABELS_FILE,
                           missing_frac=0.4, train_frac=0.8):
    raw = ingest_pandas(data_file, labels_file)
    cols = [c for c in raw.columns if c.startswith('sensor_')]
    n_train = int(len(raw) * train_frac)
    train, test = raw.iloc[:n_train].copy(), raw.iloc[n_train:].copy()
    train_c, dropped = clean_pandas(train, cols, missing_frac)
    test_c = test.drop(columns=dropped)
    keep = [c for c in train_c.columns if c.startswith('sensor_')]
    train_i = impute_pandas(train_c, train_c, keep)
    test_i = impute_pandas(train_c, test_c, keep)
    daily_i = aggregate_pandas(train_i, keep)
    return train_i, test_i, daily_i, keep

def timed(fn, repeats=3):
    return min(_time_once(fn) for _ in range(repeats))

def _time_once(fn):
    t0 = time.perf_counter()
    fn()
    return time.perf_counter() - t0

t_pandas = timed(secom_pandas_pipeline)
t_polars = timed(secom_polars_pipeline)
print(f'pandas: {t_pandas * 1000:7.1f} ms')
print(f'polars: {t_polars * 1000:7.1f} ms   ({t_pandas / t_polars:.1f}x)')


## Dask: the same pipeline, spread across partitions

This is a concepts demo, not a recommendation: 1,567 rows fit in RAM many
times over, and a real Dask deployment earns its keep on data that does not.
We split the training rows into partitions and run the pipeline as Dask
dataframe operations, so the mechanism -- a task graph over chunks, only
executed at `.compute()` -- is visible on a single laptop.

The first version below is the direct, naive port: reuse the pandas
`nunique(dropna=True)` idiom to find constant columns. Time it once before
reading on, because the result is the point.

In [ ]:
import dask.dataframe as dd

def secom_dask_pipeline_naive(raw, sensor_cols, npartitions=8,
                               missing_frac=0.4, train_frac=0.8):
    n_train = int(len(raw) * train_frac)
    train = raw.iloc[:n_train]
    ddf = dd.from_pandas(train, npartitions=npartitions)

    nunique = ddf[sensor_cols].nunique(dropna=True).compute()   # <- watch this one
    miss = ddf[sensor_cols].isna().mean().compute()
    constant = nunique[nunique <= 1].index
    high_missing = miss[miss > missing_frac].index
    keep = [c for c in sensor_cols if c not in set(constant) | set(high_missing)]
    return keep

t0 = time.perf_counter()
keep_naive = secom_dask_pipeline_naive(raw, sensor_cols, npartitions=8)
t_naive = time.perf_counter() - t0
print(f'{t_naive:.1f} s for the nunique-based version -- on 1,567 rows.')

# For scale: the identical question, asked of plain pandas.
t0 = time.perf_counter()
raw[sensor_cols].nunique(dropna=True)
t_pd_nunique = time.perf_counter() - t0
print(f'{t_pd_nunique * 1000:.0f} ms for pandas to answer the same question '
      f'({t_naive / t_pd_nunique:.0f}x faster).')


That is not a typo, and it is not this dataset being unusually hard. An exact
distinct count cannot be computed partition by partition and then summed the way
a mean can: every partition has to compare its values against every other
partition's, which is a shuffle, and Dask builds one **per column**. Six hundred
shuffles is six hundred chances to pay scheduling overhead, and the overhead,
not the arithmetic, is what you are waiting on.

The cell below measures that claim instead of asserting it. Three signatures
tell you when you are looking at overhead rather than work: the cost scales with
the number of *columns*, it gets **worse** as you add partitions, and it barely
moves when you quadruple the number of *rows*. Real computation does the
opposite on all three counts.

In [ ]:
train_rows = raw.iloc[:int(len(raw) * 0.8)]

def time_nunique(df, cols, npartitions):
    ddf = dd.from_pandas(df, npartitions=npartitions)
    t0 = time.perf_counter()
    ddf[cols].nunique(dropna=True).compute()
    return time.perf_counter() - t0

print('cost vs COLUMN count (8 partitions, 1,253 rows):')
for nc in (25, 50, 100, 200):
    dt = time_nunique(train_rows, sensor_cols[:nc], 8)
    print(f'  {nc:4d} cols: {dt:6.2f} s   ({dt / nc * 1000:5.1f} ms per column)')

print('\ncost vs PARTITION count (100 columns, same rows):')
for npart in (1, 2, 4, 8, 16):
    print(f'  {npart:2d} partitions: {time_nunique(train_rows, sensor_cols[:100], npart):6.2f} s')

print('\ncost vs ROW count (100 columns, 8 partitions):')
for mult in (1, 4):
    big = pd.concat([train_rows] * mult, ignore_index=True)
    print(f'  {len(big):6d} rows: {time_nunique(big, sensor_cols[:100], 8):6.2f} s')

# The task graph itself, for the same 100 columns.
try:
    ddf = dd.from_pandas(train_rows, npartitions=8)
    n_nunique = len(ddf[sensor_cols[:100]].nunique(dropna=True).optimize().__dask_graph__())
    n_std = len(ddf[sensor_cols[:100]].std().optimize().__dask_graph__())
    print(f'\ntasks in the graph for 100 columns: nunique={n_nunique:,}  std={n_std:,}'
          f'  ({n_nunique / n_std:.0f}x)')
except Exception as exc:   # graph introspection is not a stable public API
    print(f'\n(graph introspection unavailable on this Dask version: {exc})')


All three signatures show up: roughly a constant cost per column, a cost that
*rises* as partitions are added, and a cost that ignores the row count entirely.
Quadrupling the data does not change the runtime, which is only possible if the
runtime was never about the data.

So stop asking Dask to count distinct values. A numeric column is constant
exactly when its standard deviation is zero, and `std` is a single cheap
reduction that partitions can compute independently and combine, with no shuffle
and no per-column graph. On this data the two rules select **exactly the same
122 constant columns**, so this is a free substitution, not an approximation.
Verifying that equivalence is the part you should not skip: a faster check that
answers a slightly different question is a bug you have not noticed yet.

In [ ]:
def secom_dask_pipeline(raw, sensor_cols, npartitions=8,
                         missing_frac=0.4, train_frac=0.8):
    n_train = int(len(raw) * train_frac)
    train = raw.iloc[:n_train]
    ddf = dd.from_pandas(train, npartitions=npartitions)

    std = ddf[sensor_cols].std().compute()
    miss = ddf[sensor_cols].isna().mean().compute()
    constant = std[(std == 0) | std.isna()].index
    high_missing = miss[miss > missing_frac].index
    keep = [c for c in sensor_cols if c not in set(constant) | set(high_missing)]
    means = ddf[keep].mean().compute()

    def process_partition(part):
        out = part[['run_id', 'ts'] + keep].fillna(means)
        return out.assign(day=out['ts'].dt.date).copy()

    meta = process_partition(train.iloc[:2])
    processed = ddf.map_partitions(process_partition, meta=meta)
    daily = processed.groupby('day')[keep].mean().compute()
    return daily, keep

t0 = time.perf_counter()
daily_dask, keep_dask = secom_dask_pipeline(raw, sensor_cols, npartitions=8)
t_dask = time.perf_counter() - t0

# The std rule must select the same columns as the nunique rule, or it is not a fix.
assert set(keep_dask) == set(keep_cols), 'std-based rule disagrees with the nunique rule'

print(f'dask, std-based, 8 partitions: {t_dask * 1000:8.1f} ms')
print(f'dask, naive nunique version:   {t_naive * 1000:8.1f} ms'
      f'   ({t_naive / t_dask:.0f}x slower)')
print(f'pandas, whole pipeline:        {t_pandas * 1000:8.1f} ms')
print(f'polars, whole pipeline:        {t_polars * 1000:8.1f} ms')
print(f'\nsame {len(keep_dask)} columns kept as pandas and polars.')
print(f'dask/pandas on this data: {t_dask / t_pandas:.1f}x slower.')


## What the timings are telling you

The corrected Dask run is fast enough to sit through, but it is still several
times slower than plain pandas on this data, and that gap is the lesson, not a
bug to chase. Every Dask operation pays to build and schedule a task graph
across partitions before a single number is computed. On a table that fits in
memory, that is pure cost, because there was never anything to parallelise
across machines in the first place.

The break-even point is data that does not fit in your laptop's RAM, or a
computation heavy enough that spreading it across cores or machines pays for the
scheduling. SECOM at 1,567 rows is nowhere near it. It is here so that
partitions, a lazy task graph, and `.compute()` are things you have seen work
before you meet them at a scale where they matter.

The Dask maintainers say this themselves, in the first section of their own
DataFrame best-practices page, which is titled
[**Use Pandas**](https://docs.dask.org/en/stable/dataframe-best-practices.html):

> For data that fits into RAM, pandas can often be faster and easier to use than
> Dask DataFrame. While "Big Data" tools can be exciting, they are almost always
> worse than normal data tools while those remain appropriate.

Three things in this notebook were bugs that did not announce themselves, and
they are worth more than the timings. A quoted timestamp turned an entire column
into `NaT` without raising. Polars' schema inference committed to an integer
dtype on the strength of 100 rows and broke on row 1,458. A direct port of a
pandas idiom to Dask was 280 times slower than the alternative that computes the
same answer. Every one of them was caught by checking a number against an
expectation, which is the habit L6 turns into executable validation.

Full notes, with the pandas/Polars/Dask trade-offs and the rest of the
argument: [`notes.md`](notes.md).